# DeFiPy: Python SDK for DeFi Analytics
## Chapter 2: DeFiPy Architecture and Core Concepts

### Listing 2.1: Example: Estimating gas cost in Gwei.

In [1]:
gas_used = 21000
gas_price_gwei = 45
eth_price_usd = 3200

# Convert Gwei to ETH
fee_eth = gas_used * gas_price_gwei * 1e-9
fee_usd = fee_eth * eth_price_usd
print(f"Transaction cost ≈ ${fee_usd:.2f}")

Transaction cost ≈ $3.02


### Listing 2.2: Defining precision with UniswapExchangeData.TYPE_GWEI

In [12]:
from defipy import UniswapExchangeData

# define a Uniswap V2-style exchange data object
exchg_data = UniswapExchangeData(
    tkn0 = eth,
    tkn1 = dai,
    symbol = "LP",
    address = "0x011",
    precision = UniswapExchangeData.TYPE_GWEI
)

### Listing 2.6: Uniswap V2 Setup and Liquidity Addition

In [2]:
from defipy import ERC20, UniswapFactory, UniswapExchangeData, Join, Swap

# Step 1: Define tokens
tkn = ERC20("TKN", "0x111")
eth = ERC20("ETH", "0x999")

# Step 2:  Initialize factory
factory = UniswapFactory("ETH pool factory", "0x2")

# Step 3: Set up exchange data for V2
exch_data = UniswapExchangeData(tkn0=eth, tkn1=tkn, symbol="LP", address="0x3")

# Step 4: Deploy pool
lp = factory.deploy(exch_data)

# Step 5: Add initial liquidity
join = Join()
join.apply(lp, "user", 1000, 10000)

# Step 6: Perform swap
swap = Swap()
out = swap.apply(lp, tkn, "user", 10)

# Check reserves and liquidity
lp.summary()

Exchange ETH-TKN (LP)
Reserves: ETH = 999.00399301896, TKN = 10010.0
Liquidity: 3162.2776601683795 



### Listing 2.7: Uniswap V2 LPQuote

In [3]:
from defipy import LPQuote

# code-block 2.4

# Retrieve LP prices
p_eth = LPQuote().get_price(lp, eth)
p_tkn = LPQuote().get_price(lp, tkn)
print(f'The price of {eth.token_name} in {tkn.token_name} is {p_eth:.5f}')
print(f'The price of {tkn.token_name} in {eth.token_name} is {p_tkn:.5f}\n')

# Retrieve token settlement amount given opposing token amount
amt_eth = LPQuote(include_fee = True).get_amount(lp, eth, 1)
amt_tkn = LPQuote(include_fee = True).get_amount(lp, tkn, 1)
print(f'1 {eth.token_name} token is worth {amt_tkn:.5f} {tkn.token_name} after swap fees')
print(f'1 {tkn.token_name} token is worth {amt_eth:.5f} {eth.token_name} after swap fees\n')

# Retrieve rebased token settlement amount given amount of LP token
amt_eth = LPQuote(False).get_amount_from_lp(lp, eth, 1)
amt_tkn = LPQuote().get_amount_from_lp(lp, eth, 1)
print(f'1 LP token is worth {amt_eth:.5f} {eth.token_name}')
print(f'1 LP token is worth {amt_tkn:.5f} {tkn.token_name}\n')

# Retrieve LP token settlement amount given amount of asset token
amt_eth_lp = LPQuote(False).get_lp_from_amount(lp, eth, 1)
amt_tkn_lp = LPQuote(False).get_lp_from_amount(lp, tkn, 1)
print(f'1 {eth.token_name} token is worth {amt_eth_lp:.5f} LP tokens')
print(f'1 {tkn.token_name} token is worth {amt_tkn_lp:.5f} LP tokens')

The price of ETH in TKN is 10.01998
The price of TKN in ETH is 0.09980

1 ETH token is worth 0.09949 TKN after swap fees
1 TKN token is worth 9.97996 ETH after swap fees

1 LP token is worth 0.63078 ETH
1 LP token is worth 6.32039 TKN

1 ETH token is worth 1.58549 LP tokens
1 TKN token is worth 0.15820 LP tokens


### Listing 2.8: Uniswap V3 Setup and Liquidity Addition

In [4]:
from defipy import ERC20, UniswapFactory, UniswapExchangeData, Join, Swap

# Step 1: Define tokens and parameters
eth = ERC20("ETH", "0x93")
tkn = ERC20("TKN", "0x111")
tick_spacing = 60
fee = 3000  # 0.3% fee tier

# Step 2: Set up exchange data for V3
exch_data = UniswapExchangeData(tkn0=eth, tkn1=tkn, symbol="LP", address="0x811", version='V3', tick_spacing=tick_spacing, fee=fee)

# Step 3: Initialize factory
factory = UniswapFactory("ETH pool factory", "0x2")

# Step 4: Deploy pool
lp = factory.deploy(exch_data)

# Step 5: Add initial liquidity within tick range
lwr_tick = 22500
upr_tick = 23520
join = Join()
join.apply(lp, "user", 1000, 10000, 22500, 23520)

# Check reserves and liquidity
lp.summary()

Exchange ETH-TKN (LP)
Real Reserves:   ETH = 936.2679525254714, TKN = 9999.999999999984
Gross Liquidity: 121604.11831379162 



### Listing 2.9: Uniswap V3 LPQuote

In [5]:
# Retrieve LP prices
p_eth = LPQuote().get_price(lp, eth)
p_tkn = LPQuote().get_price(lp, tkn)
print(f'The price of {eth.token_name} in {tkn.token_name} is {p_eth:.5f}')
print(f'The price of {tkn.token_name} in {eth.token_name} is {p_tkn:.5f}\n')

# Retrieve token settlement amount given opposing token amount
amt_eth = LPQuote().get_amount(lp, eth, 1, lwr_tick, upr_tick)
amt_tkn = LPQuote().get_amount(lp, tkn, 1, lwr_tick, upr_tick)
print(f'1 {eth.token_name} token is worth {amt_tkn:.5f} {tkn.token_name}')
print(f'1 {tkn.token_name} token is worth {amt_eth:.5f} {eth.token_name}\n')

# Retrieve rebased token settlement amount given amount of LP token
amt_eth = LPQuote(False).get_amount_from_lp(lp, eth, 1, lwr_tick, upr_tick)
amt_tkn = LPQuote().get_amount_from_lp(lp, eth, 1, lwr_tick, upr_tick)
print(f'1 LP token is worth {amt_eth:.5f} {eth.token_name} after swap fees')
print(f'1 LP token is worth {amt_tkn:.5f} {tkn.token_name} after swap fees\n')

# Retrieve LP token settlement amount given amount of asset token
amt_eth_lp = LPQuote(False).get_lp_from_amount(lp, eth, 1, lwr_tick, upr_tick)
amt_tkn_lp = LPQuote(False).get_lp_from_amount(lp, tkn, 1, lwr_tick, upr_tick)
print(f'1 {eth.token_name} token is worth {amt_eth_lp:.5f} LP tokens')
print(f'1 {tkn.token_name} token is worth {amt_tkn_lp:.5f} LP tokens')

The price of ETH in TKN is 10.00000
The price of TKN in ETH is 0.10000

1 ETH token is worth 0.09970 TKN
1 TKN token is worth 9.96974 ETH

1 LP token is worth 0.01590 ETH after swap fees
1 LP token is worth 0.15850 TKN after swap fees

1 ETH token is worth 62.90124 LP tokens
1 TKN token is worth 6.28946 LP tokens


### Listing 2.10: Balancer Setup and Liquidity Addition

In [6]:
from defipy import ERC20, BalancerVault, BalancerFactory, BalancerExchangeData
from defipy import Join, Swap, AddLiquidity, RemoveLiquidity

# Step 1: Define and deposit tokens
dai = ERC20("DAI", "0x111")
weth = ERC20("WETH", "0x999")

dai.deposit(None, 10000)
weth.deposit(None, 200)

# Step 2: Setup vault and assign weights
vault = BalancerVault()
vault.add_token(dai, 10)
vault.add_token(weth, 40)

# Step 3: Create exchange data
exch_data = BalancerExchangeData(vault=vault, symbol="BSP", address="0x3")

# Step 4: Deploy the pool
bfactory = BalancerFactory("Balancer factory", "0x2")
lp = bfactory.deploy(exch_data)

# Step 5: Initialize liquidity
Join().apply(lp, "user", 100)

# Step 6: Swap tokens
Swap().apply(lp, dai, weth, "user", 500)

# Check reserves and liquidity
lp.summary()

Balancer Exchange: DAI-WETH (BSP)
Reserves: DAI = 10500, WETH = 197.58119013971427
Weights: DAI = 0.2, WETH = 0.8
Pool Shares: 100 



### Listing 2.11: Stableswap Setup and Operations

In [8]:
from defipy import ERC20, StableswapVault, StableswapFactory, StableswapExchangeData, Join, Swap

# Step 1: Define stablecoins and parameters
dai = ERC20("DAI", "0x111", 18)
usdc = ERC20("USDC", "0x222", 6)
AMPL_COEFF = 2000

# Step 2: Deposit token amounts
dai.deposit(None, 10000)
usdc.deposit(None, 20000)

# Step 3: Setup Stableswap vault and add tokens
sgrp = StableswapVault()
sgrp.add_token(dai)
sgrp.add_token(usdc)

# Step 4: Set up exchange data for Stableswap
exch_data = StableswapExchangeData(vault = sgrp, symbol="LP", address="0x011")

# Step 5: Initialize factor for Balancer
factory = StableswapFactory("Stableswap factory", "0x2")

# Step 6: Deploy pool
lp = factory.deploy(exch_data)

# Step 7: Join pool with initial liquidity
join = Join()
join.apply(lp, "user", AMPL_COEFF)

# Step 8: Perform swap
swap = Swap()
out = swap.apply(lp, dai, usdc, "user", 10)

# Check reserves and liquidity
lp.summary()

Stableswap Exchange: DAI-USDC (LP)
Reserves: DAI = 10010, USDC = 19989.996791
Liquidity: 29999.063056285642 



### Listing 2.12: Example of Uniswap V2 Module Usage

In [9]:
from defipy import ERC20, UniswapFactory, UniswapExchangeData, Join

# Step 1: Define tokens
tkn = ERC20("TKN", "0x111")
eth = ERC20("ETH", "0x999")

# Step 2: Set up exchange data for V2
exch_data = UniswapExchangeData(tkn0=eth, tkn1=tkn, symbol="LP", address="0x3")

# Step 3: Initialize factory
factory = UniswapFactory("ETH pool factory", "0x2")

# Step 4: Deploy pool
lp = factory.deploy(exch_data)

# Step 5: Add initial liquidity
join = Join()
join.apply(lp, "user", 1000, 10000)

# Check reserves and liquidity
lp.summary()  

Exchange ETH-TKN (LP)
Reserves: ETH = 1000.0, TKN = 10000.0
Liquidity: 3162.2776601683795 



### Listing 2.13: Example of Balancer Module Usage

In [10]:
from defipy import ERC20, BalancerVault, BalancerFactory, BalancerExchangeData

# Step 1: Define tokens and weights
dai = ERC20("DAI", "0x111")
weth = ERC20("WETH", "0x999")

# Step 2: Deposit token amounts
dai.deposit(None, 400000)
weth.deposit(None, 100)

# Step 3: Setup vault
vault = BalancerVault()
vault.add_token(dai, 10)  # Denormalized weight for DAI
vault.add_token(weth, 40)  # Denormalized weight for WETH

# Step 4: Set up exchange data for Balancer
exch_data = BalancerExchangeData(vault=vault, symbol="BSP", address="0x3")

# Step 5: Initialize factor for Balancer
bfactory = BalancerFactory("WETH pool factory", "0x2")

# Step 6: Deploy pool
lp = bfactory.deploy(exch_data)

# Step 7: Join pool with initial liquidity
join = Join()
join.apply(lp, "user", 100)  # Issue 100 pool shares

# Check reserves and liquidity
lp.summary()

Balancer Exchange: DAI-WETH (BSP)
Reserves: DAI = 400000, WETH = 100
Weights: DAI = 0.2, WETH = 0.8
Pool Shares: 100 

